In [2]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from PIL import ImageFile
import gc  # garbage collector to free memory

2025-10-31 12:51:42.897345: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-31 12:52:59.311299: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
print("All modules imported successfully.")

All modules imported successfully.


In [4]:
# ==============================
# Step 1: Load dataset
# ==============================
data_dir = "../../data sheets/training_set"

In [5]:
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [6]:
datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

In [7]:
batch_size = 8  # smaller batch to avoid memory overload
img_size = (224, 224)

In [8]:
generator = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="sparse",
    shuffle=False
)

Found 5000 images belonging to 5 classes.


In [9]:
# ==============================
# Step 2: Feature extraction (safe version)
# ==============================
base_model = VGG16(weights="imagenet", include_top=False, pooling="avg")

2025-10-31 13:01:09.257845: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
2025-10-31 13:01:26.999757: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 4718592 exceeds 10% of free system memory.
2025-10-31 13:01:27.005944: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 4718592 exceeds 10% of free system memory.
2025-10-31 13:01:27.630588: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 4718592 exceeds 10% of free system memory.
2025-10-31 13:01:29.679396: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 9437184 exceeds 10% of free system memory.
2025-10-31 13:01:30.411313: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 9437184 exceeds 10% of free system memory.


In [10]:
print("Extracting features batch by batch...")

Extracting features batch by batch...


In [11]:
features_list = []
labels_list = []

In [12]:
num_batches = int(np.ceil(generator.n / batch_size))

In [ ]:
for i in range(num_batches):
    x_batch, y_batch = next(generator)  # ✅ fixed line
    feat_batch = base_model.predict(x_batch, verbose=0)
    features_list.append(feat_batch)
    labels_list.append(y_batch)
    
    print(f"Processed batch {i+1}/{num_batches}")
    
    # Free memory
    del x_batch, y_batch, feat_batch
    gc.collect()

Processed batch 1/625
Processed batch 2/625
Processed batch 3/625
Processed batch 4/625
Processed batch 5/625
Processed batch 6/625
Processed batch 7/625
Processed batch 8/625
Processed batch 9/625
Processed batch 10/625
Processed batch 11/625
Processed batch 12/625
Processed batch 13/625
Processed batch 14/625
Processed batch 15/625
Processed batch 16/625
Processed batch 17/625
Processed batch 18/625
Processed batch 19/625
Processed batch 20/625
Processed batch 21/625
Processed batch 22/625
Processed batch 23/625
Processed batch 24/625
Processed batch 25/625
Processed batch 26/625
Processed batch 27/625
Processed batch 28/625
Processed batch 29/625
Processed batch 30/625
Processed batch 31/625
Processed batch 32/625
Processed batch 33/625
Processed batch 34/625
Processed batch 35/625
Processed batch 36/625
Processed batch 37/625
Processed batch 38/625
Processed batch 39/625
Processed batch 40/625
Processed batch 41/625
Processed batch 42/625
Processed batch 43/625
Processed batch 44/6

In [ ]:
# Combine all batches into arrays
features = np.vstack(features_list)
labels = np.hstack(labels_list)

: 

In [ ]:
print("Feature extraction complete.")
print("Feature shape:", features.shape)
print("Labels shape", labels.shape)

In [ ]:
# ==============================
# Step 3: Train Random Forest classifier
# ==============================
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.2, random_state=42, stratify=labels
)

NameError: name 'train_test_split' is not defined

In [ ]:
clf = RandomForestClassifier(
    n_estimators=150,
    random_state=42,
    n_jobs=-1
)

In [ ]:
print("Training Random Forest classifier...")
clf.fit(X_train, y_train)
print("Training complete.")

In [ ]:
# ==============================
# Step 4: Evaluate model
# ==============================
y_pred = clf.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print("\nAccuracy:", round(accuracy * 100, 2), "%")

In [ ]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
# ==============================
# Step 5: Predict face shape names
# ==============================
class_labels = {v: k for k, v in generator.class_indices.items()}

In [ ]:
predicted_shapes = [class_labels[int(p)] for p in y_pred]
print("\nExample Predictions (first 10):")
print(predicted_shapes[:10])